# Download BraTS 2021 (Task 1) to Google Drive

Downloads the BraTS 2021 training data (1,251 patients, 13.4 GB) from Kaggle and saves it to
`MyDrive/BrainMRI_Data/raw/brats2021/` as archives. Run it **once**; afterwards the whole team uses the Drive copy.

**Before you start (once):**
1. On kaggle.com open **Settings**, then **API**, then **Create New Token**. This downloads `kaggle.json`; open it and copy `username` and `key`.
2. In Colab click the **key icon** (Secrets) on the left and add two secrets:
   `KAGGLE_USERNAME` and `KAGGLE_KEY`. Turn on **Notebook access** for both.
3. No GPU is needed: **Runtime → Change runtime type → CPU** is fine.

Then **Runtime → Run all**. It takes roughly 15–30 minutes. Never type the Kaggle key into a cell.

In [ ]:
# 1. Connect Google Drive and read the Kaggle credentials from Colab secrets
import os
from google.colab import drive, userdata

drive.mount("/content/drive")
try:
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception as e:
    raise SystemExit("Add KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets (key icon) and allow notebook access.") from e

DATASET = "dschettler8845/brats-2021-task1"
FILES = ["BraTS2021_Training_Data.tar", "BraTS2021_00495.tar", "BraTS2021_00621.tar"]
LOCAL = "/content/dl"                                              # Colab's own fast disk
DRIVE_DIR = "/content/drive/MyDrive/BrainMRI_Data/raw/brats2021"   # the team's master copy
os.makedirs(LOCAL, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)
!df -h /content | tail -1

In [ ]:
# 2. Download the three archives from Kaggle to the Colab disk
!pip -q install kaggle
for f in FILES:
    print("Downloading", f)
    !kaggle datasets download {DATASET} -f {f} -p {LOCAL}

# Kaggle sometimes wraps a file in a .zip: unwrap it so Drive holds plain .tar files
import glob, zipfile
for z in glob.glob(f"{LOCAL}/*.zip"):
    print("Unzipping", os.path.basename(z))
    with zipfile.ZipFile(z) as zf:
        zf.extractall(LOCAL)
    os.remove(z)
!ls -lh {LOCAL}

In [ ]:
# 3. Check the archives before saving: expect 1,251 patients, 5 scans each (t1, t1ce, t2, flair, seg)
import tarfile, collections

patients = collections.defaultdict(set)
for f in FILES:
    with tarfile.open(f"{LOCAL}/{f}") as t:
        for m in t:
            if m.isfile() and m.name.endswith(".nii.gz"):
                name = os.path.basename(m.name)            # e.g. BraTS2021_00000_flair.nii.gz
                case, kind = name[:-7].rsplit("_", 1)
                patients[case].add(kind)

incomplete = {c: sorted(k) for c, k in patients.items() if k != {"t1", "t1ce", "t2", "flair", "seg"}}
print(f"Patients found: {len(patients)} (expected 1,251)")
print("Incomplete patients:", incomplete or "none")
assert len(patients) == 1251 and not incomplete, "Something is missing: do not copy, re-run step 2."

In [ ]:
# 4. Copy the archives to Drive and confirm the copies are complete
import shutil

for f in FILES:
    src, dst = f"{LOCAL}/{f}", f"{DRIVE_DIR}/{f}"
    if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
        print("Already on Drive:", f)
        continue
    print("Copying", f, "...")
    shutil.copyfile(src, dst)

for f in FILES:
    same = os.path.getsize(f"{DRIVE_DIR}/{f}") == os.path.getsize(f"{LOCAL}/{f}")
    print(f"{f:32} {os.path.getsize(f'{DRIVE_DIR}/{f}') / 1e9:6.2f} GB  {'OK' if same else 'SIZE MISMATCH'}")

In [ ]:
# 5. Finish uploading to Drive (large files keep syncing after the copy returns)
drive.flush_and_unmount()
print("Done. Open drive.google.com and check MyDrive/BrainMRI_Data/raw/brats2021 shows 3 files (about 13.4 GB).")

## Next steps

- **Share with the team:** on drive.google.com right-click `BrainMRI_Data`, then choose **Share**, add the five members as **Viewer**.
  Each member opens the shared folder and chooses **Add shortcut to Drive** so it appears under `MyDrive` in their Colab.
- **Using it for training** (start of every session): copy the archives from Drive to `/content/data`, then extract there.
  Don't extract on Drive: reading thousands of small files through Drive is very slow. See `docs/DATA_DOWNLOAD.md`, Step 3.
- **License / citation:** BraTS 2021 is CC BY 4.0; cite the BraTS 2021 paper (Baid et al., arXiv:2107.02314) and the TCIA DOI 10.7937/jc8x-9874 in the report.